In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (
    SparkSession.builder
    .appName("STEAD_XGB")
    .config("spark.driver.memory", "16g")
    .config("spark.executor.memory", "32g")
    .config("spark.executor.instances", 3)
    .getOrCreate()
)

In [2]:
parquet_path = "/expanse/lustre/projects/uci157/ysuh2/data/stead_version4"
df = spark.read.parquet(parquet_path)


In [3]:
df= df.filter(F.col("trace_category") == 'earthquake_local')

In [4]:
df.printSchema()
df.show(3)

root
 |-- trace_name: string (nullable = true)
 |-- waveform_Z: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_N: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_E: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- back_azimuth_deg: string (nullable = true)
 |-- coda_end_sample: string (nullable = true)
 |-- network_code: string (nullable = true)
 |-- p_arrival_sample: string (nullable = true)
 |-- p_status: string (nullable = true)
 |-- p_travel_sec: string (nullable = true)
 |-- p_weight: string (nullable = true)
 |-- receiver_code: string (nullable = true)
 |-- receiver_elevation_m: double (nullable = true)
 |-- receiver_latitude: double (nullable = true)
 |-- receiver_longitude: double (nullable = true)
 |-- receiver_type: string (nullable = true)
 |-- s_arrival_sample: string (nullable = true)
 |-- s_status: string (nullable = true)
 |-- s_weight: string (nullable = true)
 |-- snr_

In [5]:
feature_cols = [
    "waveform_N",
    "waveform_Z",
    "waveform_E"
]
label_col = "s_arrival_sample"

In [6]:
from pyspark.sql.types import DoubleType

# Cast features
#for c in feature_cols:
#    df = df.withColumn(c, F.col(c).cast(DoubleType()))

# Cast label
df = df.withColumn(label_col, F.col(label_col).cast(DoubleType()))

# Drop rows with nulls
#df_clean = df.dropna(subset=feature_cols + [label_col])

#print("df_clean count:", df_clean.count())

In [7]:
df.select(feature_cols + [label_col]).printSchema()


root
 |-- waveform_N: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_Z: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_E: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- s_arrival_sample: double (nullable = true)



In [11]:
from pyspark.ml.functions import array_to_vector
df = (
    df.withColumn("vec_N", array_to_vector(F.col("waveform_N")))
      .withColumn("vec_Z", array_to_vector(F.col("waveform_Z")))
      .withColumn("vec_E", array_to_vector(F.col("waveform_E")))
)

In [12]:
train_df, val_df, test_df = df.sample(withReplacement=False, fraction=0.001, seed=42).randomSplit([0.7, 0.15, 0.15], seed=42)

#print(train_df.count(), val_df.count(), test_df.count())

In [13]:
train_df.printSchema()

root
 |-- trace_name: string (nullable = true)
 |-- waveform_Z: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_N: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- waveform_E: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- back_azimuth_deg: string (nullable = true)
 |-- coda_end_sample: string (nullable = true)
 |-- network_code: string (nullable = true)
 |-- p_arrival_sample: string (nullable = true)
 |-- p_status: string (nullable = true)
 |-- p_travel_sec: string (nullable = true)
 |-- p_weight: string (nullable = true)
 |-- receiver_code: string (nullable = true)
 |-- receiver_elevation_m: double (nullable = true)
 |-- receiver_latitude: double (nullable = true)
 |-- receiver_longitude: double (nullable = true)
 |-- receiver_type: string (nullable = true)
 |-- s_arrival_sample: double (nullable = true)
 |-- s_status: string (nullable = true)
 |-- s_weight: string (nullable = true)
 |-- snr_

In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBRegressor

# ---------------------------
# Assemble features
# ---------------------------
assembler = VectorAssembler(
    inputCols=["vec_N", "vec_E", "vec_Z"],
    outputCol="features"
)

# ---------------------------
# Distributed XGBoost
# num_workers MUST match your Spark executors
# You have 3 executors → num_workers=3
# ---------------------------
xgb = SparkXGBRegressor(
    features_col="features",
    label_col=label_col,
    prediction_col="prediction",
    num_workers=3,
    max_depth=8,
    eta=0.1,
    objective="reg:squarederror",
    eval_metric="rmse"
)

pipeline = Pipeline(stages=[assembler, xgb])

# ---------------------------
# Train distributed model
# ---------------------------
model = pipeline.fit(train_df)


2026-05-18 22:19:36,109 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 3 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'max_depth': 8, 'eta': 0.1, 'eval_metric': 'rmse', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-18 22:43:03,712 INFO XGBoost-PySpark: _fit Finished xgboost training!


In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

mae_eval = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="mae"
)

train_pred = model.transform(train_df)
val_pred = model.transform(val_df)
test_pred = model.transform(test_df)

print("=== RMSE ===")
print("Train:", rmse_eval.evaluate(train_pred))
print("Val:  ", rmse_eval.evaluate(val_pred))
print("Test: ", rmse_eval.evaluate(test_pred))

print("=== MAE ===")
print("Train:", mae_eval.evaluate(train_pred))
print("Val:  ", mae_eval.evaluate(val_pred))
print("Test: ", mae_eval.evaluate(test_pred))


=== RMSE ===
Train: 1.0545980368471832
Val:   445.1195874692297
Test:  431.35559602907716
=== MAE ===
Train: 0.23962408810177196
Val:   326.8477227598249
Test:  329.7098101348878


In [16]:
print("=== Train examples ===")
train_pred.select(label_col, "prediction").show(5, truncate=False)

print("=== Validation examples ===")
val_pred.select(label_col, "prediction").show(5, truncate=False)

print("=== Test examples ===")
test_pred.select(label_col, "prediction").show(5, truncate=False)


=== Train examples ===
+----------------+------------------+
|s_arrival_sample|prediction        |
+----------------+------------------+
|1053.0          |1053.0167236328125|
|936.0           |936.075927734375  |
|853.497         |853.4898681640625 |
|1169.0          |1168.8974609375   |
|915.0           |915.1146850585938 |
+----------------+------------------+
only showing top 5 rows

=== Validation examples ===
+----------------+------------------+
|s_arrival_sample|prediction        |
+----------------+------------------+
|1454.0          |1302.8370361328125|
|1595.0          |1963.4527587890625|
|734.927         |1069.3912353515625|
|739.239         |1248.4945068359375|
|965.831         |1153.6842041015625|
+----------------+------------------+
only showing top 5 rows

=== Test examples ===
+----------------+------------------+
|s_arrival_sample|prediction        |
+----------------+------------------+
|2146.7          |1311.470703125    |
|1654.65         |1410.8087158203125|
|19